## Notebook Guide

This notebook builds two models and analyzes their drivers:

- LASSO on `uber_commission` (absolute commission)
  - Pipeline: categorical index+OHE (low-card), hashed locations (high-card), scaled numerics → linear with L1
  - Split: year-based (2023 train, 2024 test; subsampled earlier for compute)
  - Goal: interpretable coefficients; post-hoc grouping of location signals

- Random Forest on `y_cpm = uber_commission / trip_miles` (commission per mile)
  - Pipeline: leakage-aware feature set, indexed categoricals, numerics assembled
  - Split: 5% year-based (2023 train, 2024 test)
  - RF params: e.g., numTrees≈120, maxDepth≈12, maxBins=512 (tuned for high-card)
  - Goal: feature importance; post-hoc aggregation (airports overall and per airport)

Run order
1) Spark session + data load
2) LASSO pipeline & evaluation
3) RF-CPM pipeline & evaluation
4) Post-hoc analysis cells (coefficients and importances aggregation)

Notes
- Avoid rerunning heavy cells unless necessary (especially RF with high-card categoricals)
- All post-hoc aggregation cells are read-only to the trained models


# Model Training 

## Preparation

In [1]:
# High-memory SparkSession setup (limited CPU for 16-core machines)
# Run this cell once at the start of the notebook
import os
# Limit BLAS/NumPy threads BEFORE creating Spark to avoid oversubscription
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

from pyspark.sql import SparkSession

# Stop an existing session if present to apply new configs
try:
    spark.stop()
except Exception:
    pass

spark = (
    SparkSession.builder.appName("HighMem-Limited")
    .master("local[8]")                         # use ~6 cores on a 16-core machine
    .config("spark.task.cpus", "2")             # each task uses 2 CPUs -> fewer concurrent tasks
    .config("spark.sql.shuffle.partitions", "160")
    .config("spark.default.parallelism", "120")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .config("spark.driver.memory", "16g")       # avoid starving the OS
    .getOrCreate()
)

import pyspark
print("Spark version:", pyspark.__version__)
print("Spark UI:", spark.sparkContext.uiWebUrl)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/08/31 16:07:09 WARN Utils: Your hostname, yc, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
25/08/31 16:07:09 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/08/31 16:07:10 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 4.0.0
Spark UI: http://10.255.255.254:4040


In [2]:

try:
    df
except NameError:
    root = "./lake/comprehensive"
    df = (spark.read
          .option("recursiveFileLookup","true")
          .option("pathGlobFilter","*.parquet")
          .parquet(f"{root}/corrected_profitability_2023_*"))
    # Optionally union 2024 as well:
    df_2024 = (spark.read
               .option("recursiveFileLookup","true")
               .option("pathGlobFilter","*.parquet")
               .parquet(f"{root}/corrected_profitability_2024_*"))
    df = df.unionByName(df_2024, allowMissingColumns=True)
    print("rows:", df.count(), "cols:", len(df.columns))

25/08/31 16:07:14 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: ./lake/comprehensive/corrected_profitability_2023_*.
java.io.FileNotFoundException: File lake/comprehensive/corrected_profitability_2023_* does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:917)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1238)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:907)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.FileStreamSink$.hasMetadata(FileStreamSink.scala:56)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:381)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala

rows: 290096301 cols: 46


We subsample the training and testing data due to computational resource constraints. A stratified subsampling is conducted for 50% of the 2023 trainning data, and 10% of the 2024 testing data.

In [3]:
# Year-based split: 2023 -> train (subsample), 2024 -> test (subsample)
from pyspark.sql import functions as F

# Ensure year + month keys
# Drop rows with N/A boroughs early
df = df.filter((F.col("PU_Borough") != "N/A") & (F.col("DO_Borough") != "N/A"))
df = df.withColumn("year", F.year("pickup_date"))
df = df.withColumn("ym", F.date_format("pickup_date","yyyy-MM"))

seed = 42
train_frac = 0.50   # 50% per month from 2023
test_frac  = 0.10   # 10% per month from 2024

months23 = [r[0] for r in df.filter(F.col("year")==2023).select("ym").distinct().collect()]
months24 = [r[0] for r in df.filter(F.col("year")==2024).select("ym").distinct().collect()]

df_train = (df.filter(F.col("year")==2023)
              .sampleBy("ym", {m: train_frac for m in months23}, seed)
              .drop("ym"))
df_test  = (df.filter(F.col("year")==2024)
              .sampleBy("ym", {m: test_frac for m in months24}, seed)
              .drop("ym"))

print("train:", df_train.count(), "test:", df_test.count())


train: 64900884 test: 14792435


train: 64900884 test: 14792435

In [4]:
from pyspark.sql import functions as F
import pandas as pd

def get_summary(sdf, col, probs=(0.01, 0.50, 0.95, 0.98, 0.99), rel_err=0.01):
    """
    Returns (percentiles_df, stats_series).
    - percentiles_df: rows for each percentile (p5, p50, p90, p95, p99)
    - stats_series: n, mean, sd, min, max
    """
    col_ = F.col(col).cast("double")
    sdf_num = sdf.select(col_.alias(col)).where(F.col(col).isNotNull())

    # Percentiles
    qs = sdf_num.approxQuantile(col, list(probs), rel_err)
    pct_df = pd.DataFrame({
        "percentile": [f"p{int(p*100)}" for p in probs],
        col: qs
    })

    # Basic stats
    ag = (sdf_num
          .agg(F.count("*").alias("n"),
               F.mean(col).alias("mean"),
               F.stddev_samp(col).alias("sd"),
               F.min(col).alias("min"),
               F.max(col).alias("max"))
          .collect()[0].asDict())
    stats = pd.Series(ag, name=f"{col}_stats")

    return pct_df, stats

def tail_share_pct(sdf, col, threshold):
    """Percent of rows with col >= threshold (safe for large dfs)."""
    col_ = F.col(col).cast("double")
    sdf_num = sdf.select(col_.alias(col)).where(F.col(col).isNotNull())
    n = sdf_num.count()
    if n == 0:
        return 0.0
    n_tail = sdf_num.where(F.col(col) >= float(threshold)).count()
    return (n_tail / n) * 100.0


In [53]:
# Train (2023, 50%) and Test (2024, 10%)
lasso_train_pct, lasso_train_stats = get_summary(df_train, "uber_commission")
lasso_test_pct,  lasso_test_stats  = get_summary(df_test,  "uber_commission")

# Combine into one table
lasso_summary = (lasso_train_pct
                 .merge(lasso_test_pct, on="percentile", suffixes=("_train", "_test")))
print(lasso_summary)
print("\nTrain stats:\n", lasso_train_stats.round(3))
print("\nTest stats:\n",  lasso_test_stats.round(3))




  percentile  uber_commission_train  uber_commission_test
0         p1                   0.00                  0.00
1        p50                   4.17                  4.46
2        p95                  16.64                 20.59
3        p98                  22.74                 28.99
4        p99                1391.37               1199.39

Train stats:
 n       6.490393e+07
mean    5.971000e+00
sd      6.186000e+00
min     0.000000e+00
max     1.391370e+03
Name: uber_commission_stats, dtype: float64

Test stats:
 n       1.479200e+07
mean    6.952000e+00
sd      7.922000e+00
min     0.000000e+00
max     1.199390e+03
Name: uber_commission_stats, dtype: float64


In [5]:
from pyspark import StorageLevel
p_lo, p_hi = 0.01, 0.98
lo_comm, hi_comm = 0.00, 22.74
lo_comm2, hi_comm2 = 0.00,28.99
print(f"LASSO commission thresholds from train: p{int(p_lo*100)}={lo_comm:.3f}, p{int(p_hi*100)}={hi_comm:.3f}")
# 2) Filter train/test using the SAME thresholds
df_train_lasso = (df_train
                  .filter((F.col("uber_commission") >= lo_comm) & (F.col("uber_commission") <= hi_comm))
                  .persist(StorageLevel.MEMORY_AND_DISK))

df_test_lasso  = (df_test
                  .filter((F.col("uber_commission") >= lo_comm2) & (F.col("uber_commission") <= hi_comm2))
                  .persist(StorageLevel.MEMORY_AND_DISK))
print("LASSO | counts  before/after")
print("train:", df_train.count(), "->", df_train_lasso.count())
print("test :", df_test.count(),  "->", df_test_lasso.count())

LASSO commission thresholds from train: p1=0.000, p98=22.740
LASSO | counts  before/after


25/08/31 16:08:28 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


train: 64900884 -> 63455333


test : 14792435 -> 14432214


## LASSO (GLM with L1 regularization)




### LASSO section notes

Purpose
- Predict `uber_commission` for interpretability and directional insight.

Preprocessing
- Categorical (low-card): `StringIndexer` → `OneHotEncoder(dropLast=True)`
- Categorical (high-card locations): hashed representation to avoid huge OHE
- Numeric: assembled and `StandardScaler(withMean=False, withStd=True)`
- Final features: `[one-hot lows] + loc_hashed + num_scaled`

Leakage policy
- Exclude any columns that directly encode commission/revenue or per-unit monetary fields
- Use train-only fitting for indexers/encoders; transform test without refitting

Outputs
- Saved pipeline at `lake/models/glm_lasso_pipeline`
- Coefficients inspection and post-hoc grouping (e.g., airports/boroughs)

In [6]:

from pyspark.ml import Pipeline
from pyspark.sql import functions as F
from pyspark.ml.feature import StringIndexer, OneHotEncoder, FeatureHasher, VectorAssembler, StandardScaler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

# GLM pipeline: one-hot + hash + scale (no leakage)

low = [c for c in ["pickup_day_of_week","pickup_hour","PU_Borough","DO_Borough","trip_cbd_type","season","rush_hour_label","time_period"] if c in df.columns]
high = [c for c in ["PULocationID","DOLocationID","PU_Zone","DO_Zone"] if c in df.columns]

# Index + OHE for low-card
idx_low = [StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep").setStringOrderType("alphabetDesc") for c in low]
ohe_low = [OneHotEncoder(inputCol=f"{c}_idx", outputCol=f"{c}_oh", dropLast=True) for c in low]

# Hash for high-card: expects *_str columns to exist on the dataset passed to fit/transform
hasher = FeatureHasher(inputCols=[f"{c}_str" for c in high], outputCol="loc_hashed", numFeatures=2**16)

# Drop-leakage numeric features and scale
label_col = "uber_commission"
leak_cols = [
    "uber_take_rate",
    "uber_revenue_per_mile",
    "uber_revenue_per_minute",
    "driver_efficiency",
    "base_passenger_fare",
    "driver_pay",
    "pickup_year",
    "year",
]

numeric = [
    c for c, t in df.dtypes
    if t in ("double", "int", "bigint", "float", "long")
    and c not in set(low + high + [label_col] + leak_cols)
]
vec_num = VectorAssembler(inputCols=numeric, outputCol="num_vec")
scaler = StandardScaler(inputCol="num_vec", outputCol="num_scaled", withMean=False, withStd=True)

# Assemble final features
assembler = VectorAssembler(
    inputCols=[*(f"{c}_oh" for c in low), "loc_hashed", "num_scaled"],
    outputCol="features"
)

# LASSO (L1): elasticNetParam=1.0
lr = LinearRegression(featuresCol="features", labelCol=label_col, regParam=0.1, elasticNetParam=1.0)

pipeline_glm = Pipeline(stages=idx_low + ohe_low + [hasher, vec_num, scaler, assembler, lr])

In [7]:
from pyspark.sql import functions as F

def add_str_cols(d, cols):
    for c in cols:
        if c in d.columns:
            d = d.withColumn(f"{c}_str", F.col(c).cast("string"))
    return d

# high list must match your hasher's inputCols (e.g., ["PULocationID","DOLocationID","PU_Zone","DO_Zone"])
df_train1 = add_str_cols(df_train_lasso, high)
df_test1  = add_str_cols(df_test_lasso, high)

outlier analysis on sampled data

In [9]:


spark.conf.set("spark.sql.adaptive.enabled", "false")
model_glm = pipeline_glm.fit(df_train1)
pred_glm = model_glm.transform(df_test1)

25/08/31 16:14:33 WARN DAGScheduler: Broadcasting large task binary with size 4.0 MiB
25/08/31 16:16:31 WARN DAGScheduler: Broadcasting large task binary with size 4.0 MiB
25/08/31 16:16:33 WARN DAGScheduler: Broadcasting large task binary with size 4.0 MiB
25/08/31 16:18:48 WARN MemoryStore: Not enough space to cache rdd_227_97 in memory! (computed 113.0 MiB so far)
25/08/31 16:18:48 WARN BlockManager: Persisting block rdd_227_97 to disk instead.
25/08/31 16:18:51 WARN MemoryStore: Not enough space to cache rdd_227_105 in memory! (computed 65.0 MiB so far)
25/08/31 16:18:51 WARN BlockManager: Persisting block rdd_227_105 to disk instead.
25/08/31 16:18:52 WARN MemoryStore: Not enough space to cache rdd_227_97 in memory! (computed 113.0 MiB so far)
25/08/31 16:18:53 WARN DAGScheduler: Broadcasting large task binary with size 4.0 MiB
25/08/31 16:18:53 WARN DAGScheduler: Broadcasting large task binary with size 4.0 MiB
25/08/31 16:18:53 WARN MemoryStore: Not enough space to cache rdd_227

In [10]:
_e = RegressionEvaluator(labelCol="uber_commission", predictionCol="prediction")
print("GLM RMSE:", _e.setMetricName("rmse").evaluate(pred_glm))
print("GLM MAE :", _e.setMetricName("mae").evaluate(pred_glm))
print("GLM R2  :", _e.setMetricName("r2").evaluate(pred_glm))

25/08/31 16:28:15 WARN DAGScheduler: Broadcasting large task binary with size 4.0 MiB
25/08/31 16:28:41 WARN DAGScheduler: Broadcasting large task binary with size 4.0 MiB


GLM RMSE: 4.519240107128704


25/08/31 16:28:41 WARN DAGScheduler: Broadcasting large task binary with size 4.0 MiB
25/08/31 16:29:06 WARN DAGScheduler: Broadcasting large task binary with size 4.0 MiB
25/08/31 16:29:06 WARN DAGScheduler: Broadcasting large task binary with size 4.0 MiB


GLM MAE : 3.075067238978462


GLM R2  : 0.27082228272660214


25/08/31 16:29:31 WARN DAGScheduler: Broadcasting large task binary with size 4.0 MiB


In [11]:
# Save pipeline model (LASSO)
model_glm.write().overwrite().save("./lake/models/glm_lasso_pipeline")

Inspect and interpret LASSO model

In [12]:
# Ensure model is loaded for inspection
from pyspark.ml import PipelineModel
from pyspark.ml.feature import FeatureHasher, VectorAssembler

try:
    _ = model_glm
except NameError:
    model_path = "/home/yunka/project-1-individual-samhouhaoyang/lake/models/glm_lasso_pipeline"
    model_glm = PipelineModel.load(model_path)

# For downstream cells
pm = model_glm



In [13]:
# Build mapping: hash bin index -> candidate original tokens (best-effort)
from pyspark.ml.feature import FeatureHasher, VectorAssembler
import pyspark.sql.functions as F
from pyspark.sql.types import IntegerType

pm = model_glm
hasher_stage = next(s for s in pm.stages if isinstance(s, FeatureHasher))
hash_input_cols = hasher_stage.getInputCols()
num_hash = hasher_stage.getNumFeatures()

get_idx = F.udf(lambda v: int(v.indices[0]) if v is not None and hasattr(v, "indices") and len(v.indices) > 0 else None, IntegerType())

pairs_all = None
max_per_col = 3000
for ic in hash_input_cols:
    base = ic[:-4] if ic.endswith('_str') else ic
    vals = (df_train1
            .select(F.col(ic).alias(ic))
            .where(F.col(ic).isNotNull())
            .distinct()
            .limit(max_per_col))
    for jc in hash_input_cols:
        if jc != ic and jc not in vals.columns:
            vals = vals.withColumn(jc, F.lit(None).cast('string'))
    hashed = hasher_stage.transform(vals)
    pairs = (hashed
             .select(get_idx('loc_hashed').alias('idx'),
                     F.concat_ws('=', F.lit(base), F.col(ic)).alias('token'))
             .where(F.col('idx').isNotNull()))
    pairs_all = pairs if pairs_all is None else pairs_all.unionByName(pairs)

rows = [] if pairs_all is None else pairs_all.groupBy('idx').agg(F.collect_set('token').alias('tokens')).collect()
hash_idx_to_tokens = {int(r['idx']): sorted(list(r['tokens'])) for r in rows}

def pretty_hash_name(bin_name: str) -> str:
    # "hash_12345" -> "PULocationID=237 | DO_Zone=Midtown ..." (up to 5 tokens)
    if not bin_name.startswith("hash_"):
        return bin_name
    try:
        k = int(bin_name.split("_", 1)[1])
    except Exception:
        return bin_name
    toks = hash_idx_to_tokens.get(k, [])
    return " | ".join(toks[:5]) if toks else bin_name

In [14]:
# Inspect LASSO GLM coefficients with feature names
from pyspark.ml.feature import StringIndexerModel, StandardScalerModel, VectorAssembler, FeatureHasher

# 1) Grab trained linear model and pipeline
glm = model_glm.stages[-1]
pm = model_glm  # the fitted PipelineModel

# 2) Rebuild feature names in the same order as the assembler inputs
#    Assembler inputs were: [*(f"{c}_oh" for c in low), "loc_hashed", "num_scaled"]

# 2a) OHE names for low-card categorical features
idx_models = {m.getInputCol(): m for m in pm.stages if isinstance(m, StringIndexerModel)}
ohe_feature_names = []
for c in low:
    if c in idx_models:
        labels = list(idx_models[c].labels)
        # With handleInvalid="keep" and dropLast=True, kept bins == len(labels)
        ohe_feature_names.extend([f"{c}={lab}" for lab in labels])

# 2b) Hashed location features — take numFeatures from fitted pipeline
hasher_stage = next((s for s in pm.stages if isinstance(s, FeatureHasher)), None)
num_hash = hasher_stage.getNumFeatures() if hasher_stage is not None else 2**16
hash_feature_names = [f"hash_{i}" for i in range(num_hash)]

# 2c) Scaled numeric features — take input order from the fitted vec_num
vec_num_model = next((s for s in pm.stages if isinstance(s, VectorAssembler) and s.getOutputCol()=="num_vec"), None)
numeric_cols = list(vec_num_model.getInputCols()) if vec_num_model is not None else list(numeric)
scaler_model = next(s for s in pm.stages if isinstance(s, StandardScalerModel))
std_arr = scaler_model.std.toArray() if hasattr(scaler_model.std, "toArray") else list(scaler_model.std)
num_scaled_names = [f"{c} (scaled)" for c in numeric_cols]

feature_names = ohe_feature_names + hash_feature_names + num_scaled_names

# 3) Coefficients and intercept
coef = glm.coefficients
coef_arr = coef.toArray() if hasattr(coef, "toArray") else list(coef)
intercept = glm.intercept

print(f"Total coefficients: {len(coef_arr)}  | Named features constructed: {len(feature_names)}")
print(f"Intercept: {intercept}")

# Safety check
if len(coef_arr) != len(feature_names):
    print("[WARN] Name/coef length mismatch; continuing with min length.")

L = min(len(coef_arr), len(feature_names))

# 4) Top weights by absolute magnitude (skip zeros to reduce noise)
pairs = [(feature_names[i], float(coef_arr[i])) for i in range(L) if coef_arr[i] != 0.0]
pairs_named = [(pretty_hash_name(n), w) for (n, w) in pairs]
pairs_sorted = sorted(pairs_named, key=lambda x: abs(x[1]), reverse=True)

print("\nTop 50 non-zero coefficients (abs sorted):")
for name, w in pairs_sorted[:50]:
    print(f"{name}: {w:+.6f}")

# 5) Numeric coefficients back to original scale (undo StandardScaler withMean=False, withStd=True)
num_start = len(ohe_feature_names) + len(hash_feature_names)
print("\nNumeric coefficients (original scale):")
for i, c in enumerate(numeric_cols):
    idx = num_start + i
    if idx >= L:
        break
    w_scaled = float(coef_arr[idx])
    s = float(std_arr[i]) if i < len(std_arr) else 1.0
    w_orig = w_scaled / s if s not in (0.0, 0) else float("nan")
    print(f"{c}: scaled={w_scaled:+.6f}  |  per-unit(original)={w_orig:+.6f}")


Total coefficients: 65608  | Named features constructed: 65608
Intercept: 3.308096785478825

Top 50 non-zero coefficients (abs sorted):
trip_cbd_type=non_cbd: -1.769867
DO_Zone=LaGuardia Airport: +1.757725
DOLocationID=138: +1.757725
PULocationID=138: +1.210465
PU_Zone=LaGuardia Airport: +1.210465
DO_Borough=EWR: +1.094965
DO_Zone=Newark Airport: +1.094965
DOLocationID=1: +1.094965
DOLocationID=132: +0.793544
DO_Zone=JFK Airport: +0.793544
PU_Borough=Manhattan: +0.574673
PU_Zone=Cobble Hill | PU_Zone=Midtown Center: +0.475478
trip_duration_minutes (scaled): +0.418007
trip_time (scaled): +0.418007
avg_speed_mph (scaled): +0.396066
trip_cbd_type=outbound_cbd: +0.347382
trip_miles (scaled): +0.258637
PU_Zone=Upper East Side South: +0.254694
PULocationID=237: +0.254694
pickup_month (scaled): +0.243248
PU_Borough=Bronx: -0.239582
DOLocationID=237: +0.222864
DO_Zone=Upper East Side South: +0.222864
PULocationID=33: +0.216132
PU_Zone=Brooklyn Heights: +0.216132
DO_Borough=Bronx: -0.203013
PUL

### Aggregated view of your top-50 LASSO coefficients

#### Location groups (signed sum within group)
| Group | Signed sum | Count |
|---|---:|---:|
| Airports (DO) | +8.387433 | 7 |
| Airports (PU) | +2.420930 | 2 |
| Airports (PU+DO combined) | +10.808363 | 9 |
| Manhattan (PU) | +1.848957 | 6 |
| Brooklyn (PU) | +1.490372 | 7 |
| Manhattan (DO) | +0.677337 | 5 |
| Bronx (PU) | -0.239582 | 1 |
| Bronx (DO) | -0.203013 | 1 |
| Queens (PU) | -0.077546 | 1 |

Notes on mapping
- Airports: LaGuardia, JFK, Newark + their matching LocationIDs (138, 132, 1).
- Manhattan examples: Upper East Side South/North, Midtown Center.
- Brooklyn examples: Cobble Hill, Brooklyn Heights, DUMBO/Vinegar Hill, Williamsburg (North Side).
- Some lines paired zone and ID with the same magnitude (e.g., UES South and ID 237); both were counted into the same group.

#### By airport (PU, DO, combined)
| Airport | PU sum | PU count | DO sum | DO count | Combined sum | Combined count |
|---|---:|---:|---:|---:|---:|---:|
| LGA | +2.420930 | 2 | +3.515450 | 2 | +5.936380 | 4 |
| JFK | +0.000000 | 0 | +1.587088 | 2 | +1.587088 | 2 |
| EWR | +0.000000 | 0 | +3.284895 | 3 | +3.284895 | 3 |

#### Non-location categorical
| Feature | Coef |
|---|---:|
| trip_cbd_type=non_cbd | -1.769867 |
| trip_cbd_type=outbound_cbd | +0.347382 |

#### Numeric/time features (scaled)
| Feature | Coef |
|---|---:|
| trip_time (scaled) | +0.418007 |
| trip_duration_minutes (scaled) | +0.418007 |
| avg_speed_mph (scaled) | +0.396066 |
| trip_miles (scaled) | +0.258637 |
| pickup_month (scaled) | +0.243248 |
| pickup_day_of_week=6 | -0.006106 |

Key takeaways
- Airports dominate positively, especially on drop-off features; they’re also strong on pickup at LGA.
- Within NYC, Manhattan (PU/DO) and several Brooklyn PU zones contribute positively.
- Bronx (PU/DO) and Queens (PU) have small negative signals.
- Non-location: non-CBD trips are strongly negative; outbound_CBD is positive.
- Among numerics, duration/time/speed lead; distance and month follow.

## RandomForest

### RF-CPM section notes

Purpose
- Predict `y_cpm = uber_commission / trip_miles` to analyze commission efficiency per distance.

Preprocessing
- Build target with `trip_miles > 0.1` filter
- Leakage-aware feature list: exclude revenue/take-rate/per-unit monetary fields and raw denominators
- Categorical: `StringIndexer(handleInvalid="keep")` (no OHE needed for trees)
- Numeric + booleans: directly assembled via `VectorAssembler`

Sampling & split
- Year-based 5% split: 2023 train, 2024 test

Model
- `RandomForestRegressor(features="features", label="y_cpm")`
- High-card support via `maxBins=512`; typical `numTrees≈120`, `maxDepth≈12` (may be reduced for WSL stability)

Outputs
- Saved pipeline at `lake/models/rf_cpm_pipeline`
- Feature importances mapped back to feature names; airport and borough aggregations for insight


### model 2: based on Uber's commission per miles

In [2]:
from pyspark.sql import SparkSession

try:
    spark.stop()
except Exception:
    pass

spark = (
    SparkSession.builder.appName("WSL-safe")
    .master("local[4]")                         # fewer concurrent tasks
    .config("spark.task.cpus", "1")
    .config("spark.sql.shuffle.partitions", "64")
    .config("spark.default.parallelism", "64")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.driver.memory", "16g")        # 6–12g is WSL-friendly; avoid 40g
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
    .getOrCreate()
)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/08/31 13:50:52 WARN Utils: Your hostname, yc, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
25/08/31 13:50:52 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/08/31 13:50:52 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [6]:
# Year-based split: 2023 -> train (subsample), 2024 -> test (subsample)
from pyspark.sql import functions as F

# Ensure year + month keys
# Drop rows with N/A boroughs early
df = df.filter((F.col("PU_Borough") != "N/A") & (F.col("DO_Borough") != "N/A"))
df = df.withColumn("year", F.year("pickup_date"))
df = df.withColumn("ym", F.date_format("pickup_date","yyyy-MM"))

spark = (
    SparkSession.builder.appName("HighMem-Ryzen7700")
    .master("local[12]")                       # leave ~4 threads free for OS/other tasks
    .config("spark.task.cpus", "2")            # each task uses 2 vCPUs → ~6 concurrent tasks
    .config("spark.sql.shuffle.partitions", "96")  # tuned down (default=200); matches parallelism
    .config("spark.default.parallelism", "96")     # ~8× #cores, good balance
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .config("spark.driver.memory", "16g")      # safe: leaves ~10–12GB for OS/overhead
    .getOrCreate()
)


In [16]:
import pyspark.sql.functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark import StorageLevel

seed = 42
train_frac_rf = 0.05
test_frac_rf = 0.05

# 5% year-based base split FIRST (keeps all downstream ops small)
df_train_base = df.filter(F.col("pickup_year") == 2023).sample(False, train_frac_rf, seed=seed)
df_test_base  = df.filter(F.col("pickup_year") == 2024).sample(False, test_frac_rf, seed=seed)

# Create CPM only on the sampled splits
def with_cpm(d):
    return (d.filter(F.col("trip_miles") > 0.1)
              .withColumn("y_cpm", F.col("uber_commission") / F.col("trip_miles"))
              .withColumn("log_trip_miles", F.log(F.col("trip_miles") + F.lit(1e-6))))

df_cpm_train = with_cpm(df_train_base).persist(StorageLevel.MEMORY_AND_DISK)
df_cpm_test  = with_cpm(df_test_base).persist(StorageLevel.MEMORY_AND_DISK)


outlier analysis on the sampled data 

In [17]:
# Check the distribution of sampled data
rf_train_pct, rf_train_stats = get_summary(df_cpm_train, "y_cpm")
rf_test_pct,  rf_test_stats  = get_summary(df_cpm_test,  "y_cpm")

rf_summary = (rf_train_pct
              .merge(rf_test_pct, on="percentile", suffixes=("_train", "_test")))
print(rf_summary)
print("\nRF train stats:\n", rf_train_stats.round(6))
print("\nRF test stats:\n",  rf_test_stats.round(6))


  percentile  y_cpm_train   y_cpm_test
0         p1     0.000000     0.000000
1        p50     1.585253     1.840580
2        p95     5.437956     6.481132
3        p98     7.387543     8.622642
4        p99  1380.090909  1391.317460

RF train stats:
 n       6.487267e+06
mean    2.129246e+00
sd      2.635815e+00
min     0.000000e+00
max     1.380091e+03
Name: y_cpm_stats, dtype: float64

RF test stats:
 n       7.391616e+06
mean    2.495399e+00
sd      3.013608e+00
min     0.000000e+00
max     1.391317e+03
Name: y_cpm_stats, dtype: float64


In [18]:
# Remove extreme outliers from y_cpm (e.g. > p98 threshold)
# First compute threshold from training data
p98_ycpm, p98_ycpm2 = 7.387543, 8.622642

# Filter out values beyond this
df_cpm_train_clean = df_cpm_train.filter(F.col("y_cpm") <= p98_ycpm)
df_cpm_test_clean  = df_cpm_test.filter(F.col("y_cpm") <= p98_ycpm2)

print("Before:", df_cpm_train.count(), "After:", df_cpm_train_clean.count())
print("Before:", df_cpm_test.count(), "After:", df_cpm_test_clean.count())


Before: 6487267 After: 6324304


Before: 7391616 After: 7194529


2023 Before: 6487267 After: 6324304
                                                                                
2024 Before: 7391616 After: 7194529

In [26]:

# Build leakage-aware feature list on the sampled training frame
exclude = {
    "uber_commission","uber_take_rate","uber_revenue_per_mile","uber_revenue_per_minute",
    "driver_pay","base_passenger_fare",
    "pickup_date","on_scene_datetime","pickup_datetime","dropoff_datetime",
    "trip_miles","y_cpm","log_trip_miles","driver_efficiency", "avg_speed_mph", "trip_time", "trip_duration_minutes", "year", "PULocationID","DOLocationID"
}
string_cols = [c for c, t in df_cpm_train.dtypes if t == "string" and c not in exclude]
bool_cols   = [c for c, t in df_cpm_train.dtypes if t == "boolean" and c not in exclude]
numeric_cols = [
    c for c, t in df_cpm_train.dtypes
    if t in ("double","int","bigint") and c not in exclude
]
print("log_trip_miles" in numeric_cols ) # should be False
print("log_trip_miles" in [f"{c}_idx" for c in string_cols])  # should be False

indexers = [StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep") for c in string_cols]
assembler = VectorAssembler(
    inputCols=numeric_cols + [f"{c}_idx" for c in string_cols] + bool_cols,
    outputCol="features"
)

rf = RandomForestRegressor(
    featuresCol="features",
    labelCol="y_cpm",
    numTrees=120,
    maxDepth=12,
    maxBins=512,
    seed=seed,
)

pipe = Pipeline(stages=indexers + [assembler, rf])
model = pipe.fit(df_cpm_train_clean)

pred = model.transform(df_cpm_test_clean)
rmse = RegressionEvaluator(labelCol="y_cpm", predictionCol="prediction", metricName="rmse").evaluate(pred)
print(f"CPM RF RMSE (5% split): {rmse:.4f}")

model.write().overwrite().save("/home/yunka/project-1-individual-samhouhaoyang/lake/models/rf_cpm_pipeline")

# Clean up
df_cpm_train.unpersist()
df_cpm_test.unpersist()
spark.catalog.clearCache()

False
False


25/08/31 18:03:59 WARN DAGScheduler: Broadcasting large task binary with size 1216.6 KiB
25/08/31 18:04:51 WARN DAGScheduler: Broadcasting large task binary with size 2.2 MiB
25/08/31 18:06:10 WARN DAGScheduler: Broadcasting large task binary with size 4.0 MiB
25/08/31 18:08:02 WARN DAGScheduler: Broadcasting large task binary with size 7.1 MiB
25/08/31 18:10:40 WARN DAGScheduler: Broadcasting large task binary with size 12.7 MiB
25/08/31 18:14:17 WARN DAGScheduler: Broadcasting large task binary with size 1488.2 KiB
25/08/31 18:14:31 WARN DAGScheduler: Broadcasting large task binary with size 19.1 MiB
25/08/31 18:18:43 WARN DAGScheduler: Broadcasting large task binary with size 2.5 MiB
25/08/31 18:19:06 WARN DAGScheduler: Broadcasting large task binary with size 17.0 MiB
25/08/31 18:21:30 WARN DAGScheduler: Broadcasting large task binary with size 2.4 MiB
25/08/31 18:21:50 WARN DAGScheduler: Broadcasting large task binary with size 15.9 MiB
25/08/31 18:23:23 WARN DAGScheduler: Broadca

CPM RF RMSE (5% split): 1.4775


25/08/31 18:53:05 WARN TaskSetManager: Stage 698 contains a task of very large size (1180 KiB). The maximum recommended task size is 1000 KiB.


In [34]:
mae = RegressionEvaluator(labelCol="y_cpm", predictionCol="prediction", metricName="mae").evaluate(pred)
r2 = RegressionEvaluator(labelCol="y_cpm", predictionCol="prediction", metricName="r2").evaluate(pred)
print(f"CPM RF MAE (5% split): {mae:.4f}")
print(f"CPM RF R2 (5% split): {r2:.4f}")


CPM RF MAE (5% split): 1.0886
CPM RF R2 (5% split): 0.2839


### Inference for the Random Forest


In [27]:
from pyspark.ml import PipelineModel
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import RandomForestRegressionModel
import pyspark.sql.functions as F
from pyspark.sql import Row

cmp_pipeline_path = "lake/models/rf_cpm_pipeline"
rf_cmp_pipeline = PipelineModel.load(cmp_pipeline_path)

# 1) Grab the trained RF stage
rf_cmp_model = next(s for s in rf_cmp_pipeline.stages if isinstance(s, RandomForestRegressionModel))

# 2) Find the VectorAssembler that produced the "features" column
assembler = [s for s in rf_cmp_pipeline.stages if isinstance(s, VectorAssembler)][-1]
input_cols = assembler.getInputCols()

# 3) Map importances back to input feature names
cmp_imp = rf_cmp_model.featureImportances  # SparseVector over assembler input order
pairs = [(input_cols[i], float(cmp_imp[i])) for i in range(len(input_cols)) if cmp_imp[i] != 0.0]
pairs_sorted = sorted(pairs, key=lambda x: x[1], reverse=True)

# Print top-k
top_k = 30
for name, score in pairs_sorted[:top_k]:
    print(f"{name}: {score:.6f}")

# Optional: as Spark DataFrame
imp_df = spark.createDataFrame([Row(feature=name, importance=score) for name, score in pairs_sorted])
imp_df.orderBy(F.desc("importance")).show(top_k, truncate=False)

DO_Zone_idx: 0.243249
PU_Zone_idx: 0.238940
is_cross_borough: 0.161417
trip_cbd_type_idx: 0.146992
PU_service_zone_idx: 0.041657
DO_service_zone_idx: 0.033617
pickup_month: 0.027349
ym_idx: 0.025476
PU_Borough_idx: 0.019542
DO_Borough_idx: 0.018340
involves_airport: 0.008495
is_weekend: 0.006801
involves_manhattan: 0.006716
season_idx: 0.006073
temp_avg_calculated: 0.005546
pickup_day_of_week: 0.003203
wind_speed_kmh: 0.001162
precipitation_mm: 0.001010
weather_comfort_index: 0.000837
time_period_idx: 0.000761
shared_match_flag_idx: 0.000578
rush_hour_label_idx: 0.000534
holiday_name_idx: 0.000521
pickup_hour: 0.000376
is_holiday: 0.000244
fog_flag: 0.000214
is_hot: 0.000164
is_freezing: 0.000157
is_snow: 0.000027


+---------------------+---------------------+
|feature              |importance           |
+---------------------+---------------------+
|DO_Zone_idx          |0.24324914228166167  |
|PU_Zone_idx          |0.2389403830630171   |
|is_cross_borough     |0.16141650211353883  |
|trip_cbd_type_idx    |0.14699235877596376  |
|PU_service_zone_idx  |0.041657081105666044 |
|DO_service_zone_idx  |0.03361725242174495  |
|pickup_month         |0.02734892302908237  |
|ym_idx               |0.02547625860289298  |
|PU_Borough_idx       |0.019542095123460078 |
|DO_Borough_idx       |0.01834011357109546  |
|involves_airport     |0.008494575856376118 |
|is_weekend           |0.006801382792175007 |
|involves_manhattan   |0.006716211824699238 |
|season_idx           |0.00607314607031772  |
|temp_avg_calculated  |0.005546157837018134 |
|pickup_day_of_week   |0.0032032139531901396|
|wind_speed_kmh       |0.001162474027854767 |
|precipitation_mm     |0.001010277572457392 |
|weather_comfort_index|8.372651527

### Random Forest feature importance – aggregated view

#### By theme (sums to ~100%)
- Fine‑grained zones (PU_Zone_idx + DO_Zone_idx): 48.22%
- Macro geography & route (cross‑borough, CBD type, borough/service‑zone idx, involves_airport/manhattan): 43.68%
- Temporal (month, ym, DOW, season, time‑period, rush‑hour, holiday, hour, weekend): 7.13%
- Weather (temp, wind, precip, comfort, fog, hot/freezing/snow): 0.91%
- Matching/ops (shared_match_flag): 0.06%

#### PU vs DO location indices
- PU location indices total: 0.3001 (PU_Zone_idx 0.2389 + PU_service_zone_idx 0.0417 + PU_Borough_idx 0.0195)
- DO location indices total: 0.2952 (DO_Zone_idx 0.2432 + DO_service_zone_idx 0.0336 + DO_Borough_idx 0.0183)

Key takeaways
- Location granularity dominates (~92% combined), split roughly evenly between pickup and dropoff zone signals.
- Route structure (cross‑borough, CBD type) is a major driver within location effects.
- Time effects are material but secondary (~7%); weather is marginal (<1%).